# justoliunet on the FPGA

Sea / land / cloud classification of HYPSO-2 pixels, computed by the FPGA.
Each pixel's raw spectrum (120 L1a bands) goes into the kernel's registers;
the kernel drops 10 bands, z-scores the rest as in training, runs
1D-Justo-LiuNet and returns 3 scores. The highest score is the class:
**0 cloud, 1 land, 2 sea**.

Everything here came in `justoliunet_pynq.zip` from the build
(`build/justoliunet/<run>/`): `justoliunet.bit` + `justoliunet.hwh`
(the design), `xjustoliunet_hw.h` (register offsets), `justoliunet_pynq.py`
(the driver), test pixels and the scoring tool. Run the cells from the top.

## 1. Load the design onto the FPGA

Programs the FPGA with `justoliunet.bit` and sets its clock from
`justoliunet.hwh`. Needed once after every board boot, or after uploading a
new build.

In [ ]:
from justoliunet_pynq import JustoLiuNet, load_vectors, open_image
jl = JustoLiuNet()
jl.status()          # expect idle: True

## 2. Self-test

15 known spectra with the scores the reference model gives them. Every one
must pass before the results below mean anything.

In [ ]:
jl.selftest()

## 3. One pixel by hand

Write one spectrum into the kernel, start it, read the 3 scores back. Change
the index, or pass any 120 numbers to `jl.classify`.

In [ ]:
import numpy as np
inputs, expected = load_vectors()
i = 3
logits = jl.classify(inputs[i])
print('FPGA     ', np.round(logits, 5), '->', jl.class_name(logits))
print('expected ', np.round(expected[i], 5), '->', jl.class_name(expected[i]))

## 4. A whole image

Make the image folder on the PC (it needs the `hypso` package to read the
capture), zip it and upload the zip next to this notebook:

```bash
source fpga.sh
python3 tools/justoliunet_image.py prepare CAPTURE-l1a.nc --labels CAPTURE-l1a_labels.npy --step 8 --out aegean
fpga_zip aegean                 # -> aegean.zip
```

The FPGA classifies every pixel; `compare` then checks each answer against
the reference model and the labels, and draws the pictures.

In [ ]:
IMAGE = 'aegean.zip'          # or the name of an already unpacked folder
folder, logits = jl.classify_folder(IMAGE)

In [ ]:
!python3 justoliunet_image.py compare {folder}

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(folder / 'overview.png')))   # picture | labels | FPGA classes | mismatch (white = none)

## 5. Look at the input, pixel by pixel

The whole capture, then one pixel's spectrum and what the FPGA makes of it.
Row and column are positions in the sampled image (see `meta.json`).

In [ ]:
display(Image(filename=str(folder / 'capture_rgb.png')))

In [ ]:
import json
import matplotlib.pyplot as plt

pixels = np.loadtxt(folder / 'pixels.txt', dtype=np.float32, ndmin=2)
height, width = json.loads((folder / 'meta.json').read_text())['shape']
row, col = height // 2, width // 2
spectrum = pixels[row * width + col]
logits = jl.classify(spectrum)

plt.figure(figsize=(8, 3))
plt.plot(range(8, 118), spectrum[8:118])     # the 110 bands the network uses
plt.xlabel('band'); plt.ylabel('raw L1a value')
plt.title(f'pixel ({row}, {col}): {jl.class_name(logits)}   scores {np.round(logits, 2)}')
plt.show()

## 6. Speed

Milliseconds per pixel on the FPGA, including the register writes and reads
from Python.

In [ ]:
import time
start = time.monotonic()
jl.classify_pixels(pixels[:200], report_every=200)
print(f'{1000 * (time.monotonic() - start) / 200:.2f} ms per pixel')